In [8]:
import sys
import os
import torch
# Get absolute path of project root (one level up from current notebook)
project_root = os.path.abspath("..")

# Add to sys.path if not already
if project_root not in sys.path:
    sys.path.append(project_root)
# Cinemaudio-studio root (for tango_new when using Tango2)
cinema_studio_root = os.path.abspath(os.path.join(project_root, ".."))
if cinema_studio_root not in sys.path:
    sys.path.append(cinema_studio_root)       
print("Project root added to sys.path:", project_root)

import numpy as np
from helper.lib import get_model
from pydub import AudioSegment
import logging
from Variable.configurations import STEPS, SFX_RATE, SFX_GAIN
from Variable.configurations import TANGO2, TANGO_FLUX, ELEVEN_LABS
from helper.lib import Tango2Model
logger = logging.getLogger(__name__)

# Use notebook-friendly progress bars (tqdm.auto)
try:
    from tqdm.auto import tqdm
except ImportError:
    pass
from IPython.display import Audio, display
import soundfile as sf

def sfx_generator(prompt: str, duration_ms: int, model_name: str = TANGO2):
    """Generates a short sound effect using the specified model.

    Args:
        prompt: Text description of the sound effect.
        duration_ms: Target duration in milliseconds.
        model_name: One of "TangoFlux", "ElevenLabs", or "Tango2" (default: Tango2).
    """
    logger.info(f"Generating: '{prompt}' ({duration_ms}ms) with model={model_name}")

    duration_s = int(duration_ms / 1000.0)
    model_cls = get_model(model_name)
    audio_arr = model_cls.generate(prompt, steps=STEPS, duration=duration_s)

    if audio_arr is None :
        raise ValueError(f"Failed to generate audio for prompt: '{prompt}'. Model returned empty array.")

    # waveform = audio_arr.squeeze().cpu().numpy()

    # if waveform.size == 0:
    #     raise ValueError(f"Generated audio waveform is empty for prompt: '{prompt}'")

    # logger.debug(f"Audio clip from prompt {prompt} generated (shape: {waveform.shape})")

    # audio_bytes = (waveform * 32767 * SFX_GAIN).astype(np.int16).tobytes()
    segment = AudioSegment(
        data=audio_arr.tobytes(),
        sample_width=2,
        frame_rate=16000,
        channels=1,
    )
    return segment

# Tango2 outputs at 16 kHz; IPython.display.Audio requires rate when data is array/tensor
SFX_DISPLAY_RATE = 16000

def sfx_generator_for_batch(prompts: list[str], duration_ms: int, model_name: str = TANGO2):
    """Generate sound effects for multiple prompts. Returns list of audio (tensors or arrays)."""
    logger.info(f"Generating for batch of {len(prompts)} prompts with duration {duration_ms}ms with model={model_name}")
    duration_s = int(duration_ms / 1000.0)
    model_cls = get_model(model_name)
    audio_list = model_cls.generate_for_batch(prompts, steps=STEPS, duration=duration_s)
    return audio_list


def _to_numpy(audio_out):
    """Convert model output (tensor or array) to float32 numpy for playback/save."""
    if hasattr(audio_out, "cpu"):
        return audio_out.squeeze().cpu().numpy().astype(np.float32)
    return np.asarray(audio_out).squeeze().astype(np.float32)




test_prompts = ["Rolling thunder with lightning strikes", "Gun shots", "Explosion"]
test_duration_ms = 5000  # 5 seconds
test_model_name = TANGO2  # or TANGO_FLUX, ELEVEN_LABS

sfx_audio_list = sfx_generator_for_batch(test_prompts, test_duration_ms, model_name=test_model_name)
os.makedirs("Debug", exist_ok=True)

# Display each audio with prompt, model name, and player
for prompt, audio_out in zip(test_prompts, sfx_audio_list):
    samples = _to_numpy(audio_out)
    print(f"Generated by: {test_model_name}")
    print(f"▶ Prompt: {prompt}")
    display(Audio(samples, rate=SFX_DISPLAY_RATE))
    sf.write("Debug/test_" + prompt.replace(" ", "_") + ".wav", samples, SFX_DISPLAY_RATE)
    print(f"   Saved: Debug/test_{prompt.replace(' ', '_')}.wav\n")


Project root added to sys.path: /Users/ajitesh/Desktop/BTP/cinemaudio-studio/backend
2026-03-08 20:25:45,471 - __main__ - INFO - Generating for batch of 3 prompts with duration 5000ms with model=Tango2
2026-03-08 20:25:45,472 - helper.lib - INFO - Generating audio from Tango2 for batch of 3 prompts with duration 5 seconds


100%|██████████| 1/1 [02:00<00:00, 120.35s/it]

Generated by: Tango2
▶ Prompt: Rolling thunder with lightning strikes


   Saved: Debug/test_Rolling_thunder_with_lightning_strikes.wav

Generated by: Tango2
▶ Prompt: Gun shots


   Saved: Debug/test_Gun_shots.wav

Generated by: Tango2
▶ Prompt: Explosion


   Saved: Debug/test_Explosion.wav

